# 01 — Explore the benchmark

What do we search (the corpus), what do we ask (the questions), and are the labels usable? Run `make data` first. Example questions come from **train only**; test questions are counted but never read here.

In [1]:
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

from thairag.data import load_corpus, load_questions

pd.set_option("display.max_colwidth", 60)
RAW = Path("../data/raw")
corpus = load_corpus(RAW)
train, test = load_questions("train", RAW), load_questions("test", RAW)
ids = set(corpus.section_id)
print(f"corpus: {len(corpus)} sections, {corpus.law_name.nunique()} laws | train {len(train)} | test {len(test)}")
for name, q in [("train", train), ("test", test)]:
    labels = q.positive_ids.explode().dropna()
    print(f"{name}: {len(labels)} positive labels, {(~labels.isin(ids)).sum()} missing from corpus")

corpus: 5127 sections, 36 laws | train 8211 | test 3742
train: 12109 positive labels, 0 missing from corpus
test: 3742 positive labels, 0 missing from corpus


## Questions per law

Law of the first positive label. Share in percent.

In [2]:
per_law = pd.DataFrame({
    "sections": corpus.law_name.value_counts(),
    "train_%": train.law_name.value_counts(normalize=True) * 100,
    "test_n": test.law_name.value_counts(),
    "test_%": test.law_name.value_counts(normalize=True) * 100,
}).fillna(0).round(1).sort_values("test_%", ascending=False)
per_law.head(22)

,sections,train_%,test_n,test_%
law_name,,,,
ประมวลกฎหมายแพ่งและพาณิชย์,1842,30.9,1620.0,43.3
ประมวลรัษฎากร,361,7.6,486.0,13.0
พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535,499,12.5,294.0,7.9
พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535,231,7.7,186.0,5.0
พระราชบัญญัติธุรกิจสถาบันการเงิน พ.ศ. 2551,168,2.0,165.0,4.4
พระราชบัญญัติการประกอบกิจการพลังงาน พ.ศ. 2550,155,1.9,153.0,4.1
พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514,115,1.4,110.0,2.9
พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,156,2.4,109.0,2.9
พระราชกำหนดการประกอบธุรกิจสินทรัพย์ดิจิทัล พ.ศ. 2561,100,2.4,81.0,2.2


## Lengths

Tokens are counted with BGE-M3's own tokenizer (the embedding model we use in M2). BGE-M3 reads at most 8,192 tokens.

In [3]:
tok = AutoTokenizer.from_pretrained("BAAI/bge-m3")
n_tok = lambda texts: [len(x) for x in tok(list(texts))["input_ids"]]
lengths = pd.DataFrame({
    "section chars": corpus.text.str.len().describe(percentiles=[.5, .9, .99]),
    "section tokens": pd.Series(n_tok(corpus.text)).describe(percentiles=[.5, .9, .99]),
    "train Q tokens": pd.Series(n_tok(train.question)).describe(percentiles=[.5, .9, .99]),
    "test Q tokens": pd.Series(n_tok(test.question)).describe(percentiles=[.5, .9, .99]),
}).round(0)
lengths

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

,section chars,section tokens,train Q tokens,test Q tokens
count,5127.0,5127.0,8211.0,3742.0
mean,466.0,121.0,21.0,23.0
std,466.0,115.0,10.0,13.0
min,31.0,15.0,6.0,4.0
50%,336.0,90.0,19.0,20.0
90%,908.0,229.0,30.0,38.0
99%,2116.0,531.0,59.0,72.0
max,6696.0,1685.0,226.0,229.0


## Train has two kinds of rows

The first ~840 rows have one label and no hard negatives, and follow the same section order as test. The dataset card doesn't mention this. They cover only a few laws.

In [4]:
BLOCK = 837
kind = pd.Series(["block (first 837)" if i < BLOCK else "rest" for i in range(len(train))])
summary = train.assign(kind=kind, n_pos=train.positive_ids.map(len), n_neg=train.hard_negative_ids.map(len)).groupby("kind").agg(
    rows=("question", "size"), one_label=("n_pos", lambda s: (s == 1).mean()), no_label=("n_pos", lambda s: (s == 0).sum()),
    no_hard_neg=("n_neg", lambda s: (s == 0).mean()), laws=("law_name", "nunique")).round(2)
summary

,rows,one_label,no_label,no_hard_neg,laws
kind,,,,,
block (first 837),837,1.00,0,1.00,6
rest,7374,0.66,72,0.03,30


In [5]:
train.law_name[:BLOCK].value_counts(normalize=True).round(2)

law_name
ประมวลรัษฎากร                                           0.42
พระราชบัญญัติธุรกิจสถาบันการเงิน พ.ศ. 2551              0.20
พระราชบัญญัติการประกอบกิจการพลังงาน พ.ศ. 2550           0.18
พระราชบัญญัติภาษีเงินได้ปิโตรเลียม พ.ศ. 2514            0.13
พระราชบัญญัติการส่งเสริมการอนุรักษ์พลังงาน พ.ศ. 2535    0.07
พระราชบัญญัติวินัยการเงินการคลังของรัฐ พ.ศ. 2561        0.00
Name: proportion, dtype: float64

## Five train rows to read together

In [6]:
text = dict(zip(corpus.section_id, corpus.text))
rest = train[BLOCK:]
picks = pd.concat([
    train[:BLOCK].sample(1, random_state=42),
    rest[rest.positive_ids.map(len) == 1].sample(2, random_state=42),
    rest[rest.positive_ids.map(len) == 3].sample(1, random_state=42),
    rest[(rest.positive_ids.map(len) == 0)].sample(1, random_state=42),
])
for _, r in picks.iterrows():
    print(f"=== {r.question_id} | {len(r.positive_ids)} label(s), {len(r.hard_negative_ids)} hard negative(s)")
    print("Q:", r.question)
    for i in r.positive_ids:
        print("  +", i, "|", text[i][:260].replace("\n", " "))
    print("A:", r.answer[:350].replace("\n", " "), "\n")

=== train-00209 | 1 label(s), 0 hard negative(s)
Q: ในกรณีใดบ้าง ที่กรรมการจะพ้นจากตำแหน่งที่ไม่ได้เป็นการพ้นจากตำแหน่งตามวาระ
  + ป0008-1D-0001-13 จัตวา | ประมวลรัษฎากร มาตรา 13 จัตวา นอกจากการพ้นจากตำแหน่งตามวาระตามมาตรา 13 ตรี กรรมการซึ่งรัฐมนตรีแต่งตั้งพ้นจากตำแหน่ง เมื่อ (1) ตาย (2) ลาออก (3) รัฐมนตรีให้ออก (4) เป็นคนไร้ความสามารถหรือเสมือนไร้ความสามารถ หรือเป็นบุคคลล้มละลาย (5) ได้รับโทษจำคุกโดยคำพิพากษาถึง
A: (1) ตาย (2) ลาออก (3) รัฐมนตรีให้ออก (4) เป็นคนไร้ความสามารถหรือเสมือนไร้ความสามารถ หรือเป็นบุคคลล้มละลาย (5) ได้รับโทษจำคุกโดยคำพิพากษาถึงที่สุดให้จำคุก เว้นแต่เป็นโทษสำหรับความผิดที่ได้กระทำโดยประมาทหรือความผิดลหุโทษ คำอธิบาย ประมวลรัษฎากร มาตรา 13 จัตวา นอกจากการพ้นจากตำแหน่งตามวาระตามมาตรา 13 ตรี กรรมการซึ่งรัฐมนตรีแต่งตั้งพ้นจากตำแหน่ง เมื่อ ( 

=== train-01082 | 1 label(s), 4 hard negative(s)
Q: การหักบัญชีในบริบทของศูนย์รับฝากหลักทรัพย์หมายความว่าอย่างไร
  + ห0015-1B-0002-220 | พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535 มาตรา 220 ห้ามมิให้บุคคลใดประกอบการเป็น